# Google Geocoding CSV ツール (Colab)\n\nこのノートブックは `geocode_csv.py` を Google Colab 上で実行するためのものです。

> 実行前に `WORKFLOW_RULES.md` を確認してください（全体ルール）。


## 1) API キーを設定

In [ ]:
import os
import getpass

os.environ['GOOGLE_MAPS_API_KEY'] = getpass.getpass('Google Maps API Key: ')
print('API キーを環境変数 GOOGLE_MAPS_API_KEY に設定しました。')

## 2) 入力CSVをアップロード

In [ ]:
from google.colab import files
from pathlib import Path
import json
import shutil

WORK_ROOT = Path('/content/geocode_workdirs')
WORK_ROOT.mkdir(parents=True, exist_ok=True)

def _write_state(folder: Path, input_csv_name: str) -> None:
    state_path = folder / 'work_state.json'
    state = {
        'input_csv_name': input_csv_name,
        'output_csv_name': 'output_with_latlng.csv',
        'final_output_csv_name': 'output_finalized.csv',
    }
    state_path.write_text(json.dumps(state, ensure_ascii=False, indent=2), encoding='utf-8')

uploaded = files.upload()
print('アップロード済みファイル:', list(uploaded.keys()))

for raw_name in uploaded.keys():
    src = Path(raw_name)
    if src.suffix.lower() == '.csv':
        folder = WORK_ROOT / src.stem
        folder.mkdir(parents=True, exist_ok=True)
        dst = folder / src.name
        shutil.copy2(src, dst)
        if not (folder / 'work_state.json').exists():
            _write_state(folder, src.name)
        print(f'CSVを作業フォルダへ保存: {dst}')
    elif '/' in raw_name:
        # フォルダ再アップロード時のパス構造を復元
        dst = WORK_ROOT / raw_name
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
        print(f'フォルダファイルを復元: {dst}')

print('現在の作業フォルダ一覧:')
for d in sorted([x for x in WORK_ROOT.iterdir() if x.is_dir()]):
    print(' -', d.name)


## 3) `geocode_csv.py` を利用する準備（GitHub同期対応）
- **推奨**: GitHub リポジトリを同期して、`geocode_csv.py` を毎回同じバージョンで実行します。
- `GITHUB_REPO_URL` を設定すると、Colab上の作業ディレクトリを毎回作り直して `git clone` します（= ローカル差分や古いファイルをリセット）。
- 未設定の場合は、従来通り `geocode_csv.py` を手動アップロードして使えます。


In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

# リポジトリは固定運用
GITHUB_REPO_URL = 'https://github.com/yandonomae/vitejs-vite-gt76zzzr.git'
GITHUB_BRANCH = 'main'
SYNC_DIR = Path('/content/geocode_repo')

if SYNC_DIR.exists():
    shutil.rmtree(SYNC_DIR)

clone_cmd = [
    'git', 'clone', '--depth', '1', '--branch', GITHUB_BRANCH, GITHUB_REPO_URL, str(SYNC_DIR)
]
print('同期コマンド:', ' '.join(clone_cmd))
subprocess.run(clone_cmd, check=True)

SCRIPT_PATH = SYNC_DIR / 'geocode_csv.py'
REVIEW_SCRIPT_PATH = SYNC_DIR / 'review_fallbacks.py'
RULES_PATH = SYNC_DIR / 'WORKFLOW_RULES.md'

if not SCRIPT_PATH.exists():
    raise FileNotFoundError(f'同期先に geocode_csv.py が見つかりません: {SCRIPT_PATH}')

if RULES_PATH.exists():
    print('--- WORKFLOW_RULES.md ---')
    print(RULES_PATH.read_text(encoding='utf-8'))
    print('--- end ---')

print(f'実行スクリプト: {SCRIPT_PATH.resolve()}')


## 4) 変換を実行（ログを時刻付きファイルに保存）
ログは `logs/` 配下に `geocode_YYYYMMDD_HHMMSS.log` 形式で保存します（上書きしません）。


In [ ]:
from pathlib import Path
import json

WORK_ROOT = Path('/content/geocode_workdirs')
workdirs = sorted([p for p in WORK_ROOT.iterdir() if p.is_dir()])
if not workdirs:
    raise FileNotFoundError('作業フォルダがありません。先にCSVまたは作業フォルダをアップロードしてください。')

print('作業対象フォルダを選択してください:')
for i, d in enumerate(workdirs, start=1):
    print(f'[{i}] {d.name}')

selected = int(input('番号を入力: ').strip())
TARGET_DIR = workdirs[selected - 1]

state_path = TARGET_DIR / 'work_state.json'
if state_path.exists():
    state = json.loads(state_path.read_text(encoding='utf-8'))
else:
    csvs = sorted(TARGET_DIR.glob('*.csv'))
    if not csvs:
        raise FileNotFoundError(f'CSVが見つかりません: {TARGET_DIR}')
    state = {
        'input_csv_name': csvs[0].name,
        'output_csv_name': 'output_with_latlng.csv',
        'final_output_csv_name': 'output_finalized.csv',
    }

INPUT_CSV = str(TARGET_DIR / state['input_csv_name'])
OUTPUT_CSV = str(TARGET_DIR / state.get('output_csv_name', 'output_with_latlng.csv'))
FINAL_OUTPUT_CSV = str(TARGET_DIR / state.get('final_output_csv_name', 'output_finalized.csv'))

DISABLE_PLACES = False
PLACES_BIAS_RADIUS = 3000
SLEEP_SECONDS = 0.1
TIMEOUT_SECONDS = 10

print('選択中の作業フォルダ:', TARGET_DIR)
print('INPUT_CSV:', INPUT_CSV)
print('OUTPUT_CSV:', OUTPUT_CSV)
print('FINAL_OUTPUT_CSV:', FINAL_OUTPUT_CSV)


In [ ]:
import os
import subprocess
from datetime import datetime
from pathlib import Path

API_KEY = os.environ.get('GOOGLE_MAPS_API_KEY', '').strip()
if not API_KEY:
    raise ValueError('GOOGLE_MAPS_API_KEY が未設定です。最初のセルを実行してください。')

if 'TARGET_DIR' not in globals():
    raise RuntimeError('先に作業フォルダ選択セルを実行してください。')

def _script_supports_option(script_path: Path, option: str) -> bool:
    probe = subprocess.run(['python', str(script_path), '--help'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, check=False)
    return option in probe.stdout

logs_dir = Path(TARGET_DIR) / 'logs'
logs_dir.mkdir(parents=True, exist_ok=True)
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
log_path = logs_dir / f'geocode_{timestamp}.log'

cmd = ['python', str(SCRIPT_PATH), INPUT_CSV, OUTPUT_CSV]
if _script_supports_option(SCRIPT_PATH, '--places-bias-radius'):
    cmd.extend(['--places-bias-radius', str(PLACES_BIAS_RADIUS)])
if _script_supports_option(SCRIPT_PATH, '--sleep'):
    cmd.extend(['--sleep', str(SLEEP_SECONDS)])
if _script_supports_option(SCRIPT_PATH, '--timeout'):
    cmd.extend(['--timeout', str(TIMEOUT_SECONDS)])
if DISABLE_PLACES and _script_supports_option(SCRIPT_PATH, '--disable-places'):
    cmd.append('--disable-places')

print('実行コマンド:', ' '.join(cmd))
print('ログファイル:', log_path)
print('進捗表示: n/total, 1件平均秒, 残り推定, 予想到達時刻')

with log_path.open('w', encoding='utf-8') as logf:
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end='')
        logf.write(line)

return_code = proc.wait()
print(f'終了コード: {return_code}')
if return_code != 0:
    raise RuntimeError(f'geocode_csv.py の実行に失敗しました。ログを確認してください: {log_path}')

print(f'完了。ログ保存先: {log_path}')


## 5) LOW_PLACE_CONFIDENCE のうち距離50m以下を一括で Places 採用（任意）


In [ ]:
import csv
from pathlib import Path

AUTO_REVIEW_ENABLED = True
AUTO_REVIEW_MAX_DISTANCE_M = 50.0

if not AUTO_REVIEW_ENABLED:
    print('一括自動採用をスキップしました。')
else:
    in_path = Path(OUTPUT_CSV)
    out_path = Path(FINAL_OUTPUT_CSV)

    if not in_path.exists():
        raise FileNotFoundError(f'入力CSVが見つかりません: {in_path}')

    source_path = out_path if out_path.exists() else in_path
    with source_path.open('r', encoding='utf-8-sig', newline='') as f:
        reader = csv.DictReader(f)
        headers = list(reader.fieldnames or [])
        rows = list(reader)

    required_cols = ['採用方式', 'フォールバック理由', '緯度', '経度', '正規化住所', '候補住所(採用)', '候補緯度(採用)', '候補経度(採用)', 'places_geocode_distance_m']
    for col in required_cols:
        if col not in headers:
            raise ValueError(f'必要列がありません: {col}')

    changed = 0
    for row in rows:
        method = (row.get('採用方式', '') or '').strip()
        reason = (row.get('フォールバック理由', '') or '').strip()
        if method not in {'geocode_fallback', 'places_auto_review'}:
            continue
        if not reason.startswith('LOW_PLACE_CONFIDENCE:'):
            continue

        cand_lat = (row.get('候補緯度(採用)') or '').strip()
        cand_lng = (row.get('候補経度(採用)') or '').strip()
        if not cand_lat or not cand_lng:
            continue

        try:
            distance_m = float((row.get('places_geocode_distance_m') or '').strip())
        except ValueError:
            continue

        if distance_m <= AUTO_REVIEW_MAX_DISTANCE_M:
            row['緯度'] = cand_lat
            row['経度'] = cand_lng
            row['正規化住所'] = row.get('候補住所(採用)', '')
            row['採用方式'] = 'places_auto_review'
            row['信頼度'] = 'auto_override'
            row['フォールバック理由'] = f'AUTO_PLACE_OVERRIDE:distance_le_{int(AUTO_REVIEW_MAX_DISTANCE_M)}m'
            changed += 1

    with out_path.open('w', encoding='utf-8-sig', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=headers)
        writer.writeheader()
        writer.writerows(rows)

    print(f'一括採用件数: {changed}件 (閾値: {AUTO_REVIEW_MAX_DISTANCE_M}m)')
    print(f'保存先: {out_path}')



## 6) geocode_fallback をNotebook上で対話レビュー（任意）


In [ ]:
import csv
from pathlib import Path

REVIEW_IN_NOTEBOOK = True

if not REVIEW_IN_NOTEBOOK:
    print('レビューをスキップしました。')
    DOWNLOAD_CSV = OUTPUT_CSV
else:
    in_path = Path(OUTPUT_CSV)
    out_path = Path(FINAL_OUTPUT_CSV)

    if not in_path.exists():
        raise FileNotFoundError(f'入力CSVが見つかりません: {in_path}')

    source_path = out_path if out_path.exists() else in_path
    with source_path.open('r', encoding='utf-8-sig', newline='') as f:
        reader = csv.DictReader(f)
        headers = list(reader.fieldnames or [])
        rows = list(reader)

    required_cols = ['採用方式', 'フォールバック理由', '緯度', '経度', '正規化住所', '候補住所(採用)', '候補緯度(採用)', '候補経度(採用)']
    for col in required_cols:
        if col not in headers:
            raise ValueError(f'必要列がありません: {col}')

    targets = [
        i for i, r in enumerate(rows)
        if (r.get('採用方式', '').strip() in {'geocode_fallback', 'places_manual_review'} and r.get('フォールバック理由', '').strip().startswith('LOW_PLACE_CONFIDENCE:'))
    ]
    print(f'レビュー対象: {len(targets)}件 (読み込み元: {source_path.name})')

    resume_mode = input('再開方法 [r=最後にplaces採用した行の次から / a=先頭から]: ').strip().lower() or 'r'
    start_pos = 0
    if resume_mode == 'r':
        last_place_idx = -1
        for i, r in enumerate(rows):
            if r.get('採用方式', '').strip() == 'places_manual_review':
                last_place_idx = i
        if last_place_idx >= 0:
            for pos, idx in enumerate(targets):
                if idx > last_place_idx:
                    start_pos = pos
                    break
            else:
                start_pos = len(targets)
        print(f'再開位置: {start_pos + 1}/{len(targets)}')

    history = []
    stop_requested = False
    pos = start_pos

    while pos < len(targets):
        idx = targets[pos]
        row = rows[idx]
        snapshot = dict(row)

        print('\n' + '=' * 70)
        print(f'[{pos + 1}/{len(targets)}] 行番号(ヘッダー除く): {idx + 2}')
        print(f"店名(元csv): {row.get('店の名前', '')}")
        print(f"候補(places): {row.get('候補店名(採用)', '')}")
        print(f"住所(元csv): {row.get('住所', '')}")
        print(f"候補住所(places): {row.get('候補住所(採用)', '')}")
        print('--- 付帯情報 ---')
        print(f"現在採用(geocode): ({row.get('緯度', '')}, {row.get('経度', '')}) / {row.get('正規化住所', '')}")
        print(f"候補座標: ({row.get('候補緯度(採用)', '')}, {row.get('候補経度(採用)', '')})")
        print(f"候補スコア: {row.get('候補スコア', '')} / 距離m: {row.get('places_geocode_distance_m', '')}")

        cand_lat = (row.get('候補緯度(採用)') or '').strip()
        cand_lng = (row.get('候補経度(採用)') or '').strip()
        if not cand_lat or not cand_lng:
            print('候補座標がないため geocode 維持。')
            pos += 1
            continue

        cmd = input('選択 [p=places採用 / g=geocode維持 / b=1つ戻る / q=終了して保存]: ').strip().lower()
        if cmd == 'p':
            history.append((idx, snapshot))
            row['緯度'] = cand_lat
            row['経度'] = cand_lng
            row['正規化住所'] = row.get('候補住所(採用)', '')
            row['採用方式'] = 'places_manual_review'
            row['信頼度'] = 'manual_override'
            row['フォールバック理由'] = 'MANUAL_PLACE_OVERRIDE'
            print('→ Placesを採用')
            pos += 1
            continue
        if cmd == 'g':
            history.append((idx, snapshot))
            if row.get('採用方式', '').strip() == 'places_manual_review':
                # 再判断でgeocodeに戻したい場合
                row['採用方式'] = 'geocode_fallback'
                row['信頼度'] = 'low'
                row['フォールバック理由'] = 'LOW_PLACE_CONFIDENCE:low'
            print('→ Geocode維持')
            pos += 1
            continue
        if cmd == 'b':
            if not history:
                print('戻れる履歴がありません。')
                continue
            prev_idx, prev_snapshot = history.pop()
            rows[prev_idx].update(prev_snapshot)
            prev_pos = targets.index(prev_idx)
            pos = prev_pos
            print(f'→ 1つ戻りました（{pos + 1}/{len(targets)}）')
            continue
        if cmd == 'q':
            print('→ 途中終了して保存')
            stop_requested = True
            break
        print('p / g / b / q を入力してください。')

    with out_path.open('w', encoding='utf-8-sig', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=headers)
        writer.writeheader()
        writer.writerows(rows)

    print(f'レビュー済みCSVを保存しました: {out_path}')
    if stop_requested:
        print('次回は同じ作業フォルダを再アップロードして再開できます。')
    DOWNLOAD_CSV = str(out_path)


## 7) 出力CSVをダウンロード


In [ ]:
from google.colab import files
import shutil
from pathlib import Path

if 'TARGET_DIR' not in globals():
    raise RuntimeError('先に作業フォルダ選択セルを実行してください。')

mode = input('ダウンロード種別 [1=最終アウトプットのみ / 2=作業フォルダ全体(zip)]: ').strip() or '1'
if mode == '2':
    zip_base = str(TARGET_DIR)
    archive_path = shutil.make_archive(zip_base, 'zip', root_dir=TARGET_DIR.parent, base_dir=TARGET_DIR.name)
    print('作業フォルダzipをダウンロード:', archive_path)
    files.download(archive_path)
else:
    path = DOWNLOAD_CSV if 'DOWNLOAD_CSV' in globals() else OUTPUT_CSV
    print('CSVをダウンロード:', path)
    files.download(path)
